## Week 2 Day 2 - Orchestration

Our first Agentic Framework project!!

### Part 1: Email Setup

### Part 2: Orchestrating by code

### Part 3: Orchestrating by LLMs

- 3a: via Tools
- 3b: via Handoffs

## Part 1: Email Setup

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Sending Emails</h2>
            <span style="color:#ff7800;">We are going to write an agent that sends emails. The best way to do this is with an email provider
            like SendGrid or Resend. But this requires a fair amount of heavy-lifting: you need to send from a proper email host where you own the domain and you can prove it with DNS records. It's quite a hassle.<br/><br/>
            So we're going to do it the free & simple way: using SMTP server configuration to send direct from your email. It's an easy setup, but it's not as powerful (eg you can't receive emails). If you'd like to take this further, use SendGrid or Resend instead..<br/><br/>
            And actually the email send is optional. We're doing this to demonstrate an Agent sending emails. The important part is the Agent work, not the email sending. Feel free to swap out the function for a Pushover push if you'd prefer.
            </span>
        </td>
    </tr>
</table>

## Setting up to send emails from your SMTP server

### STEP 1: Determining your SMTP Server

Either google or ask ChatGPT / Claude for the SMTP server for your email. Here are some common ones. Some email providers might not have SMTP servers enabled (eg Microsoft 365 for work/school).

Google: smtp.gmail.com  
Outlook.com / Hotmail / Live: smtp-mail.outlook.com  
Microsoft 365: smtp.office365.com  
iCloud Mail: smtp.mail.me.com  

Add to your .env file:  

`EMAIL_SMTP_SERVER=xxxx`

### STEP 2: Obtain an app specific password

Google how to do this for your email provider. For gmail, you need to have 2-step verification on. Then visit this page:

https://myaccount.google.com/apppasswords

Give it any name; copy the password and add it to your .env file, removing the spaces that it adds. (It should be 16 characters, no spaces.)

`EMAIL_APP_PASSWORD=xxxx`

### STEP 3: Add in your email address:

`EMAIL_ADDRESS=xxx`

Remember to Save the .env file!

In [ ]:
from dotenv import load_dotenv
import requests
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)

gemini_api_key = os.getenv("GOOGLE_API_KEY")
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(api_key=gemini_api_key, base_url=gemini_base_url)
gemini_model = OpenAIChatCompletionsModel(openai_client=gemini_client, model="gemini-2.5-flash-lite")

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_base_url = "https://openrouter.ai/api/v1"
openrouter_client = AsyncOpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)
openrouter_model = OpenAIChatCompletionsModel(openai_client=openrouter_client, model="cohere/north-mini-code:free")

MODEL_NAME = "gpt-5.4-mini"

In [ ]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

In [3]:
# Here we go

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [5]:
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

### If this didn't work, then uncomment the below so that we don't use emails

In [ ]:
# USE_EMAIL = False

### Our fallback strategy - send a push

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [48]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

### OK Now everything should work!

In [49]:
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

## Agent Orchestration

There are 2 models for Agent Orchestration; by code and by LLMs.

By code: more predictable and deterministic.

By LLMs: more powerful.

An excellent write-up is here:

https://openai.github.io/openai-agents-python/multi_agent/

We will start with by Code.

## Part 2: Orchestrating by Code

In [50]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [ ]:
send_email_tool.params_json_schema

In [44]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

decision1 = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

decision2 = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")

In [45]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=openrouter_model)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=openrouter_model)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=openrouter_model)
sales_picker = Agent(name="Sales Picker", instructions=decision1, model=openrouter_model)
sales_sender = Agent(name="Sales sender", instructions=decision2, model=openrouter_model, tools=[send_email_tool], model_settings=require_tool)

In [ ]:
message = "write a cold sales email"

results = await asyncio.gather(
    Runner.run(sales_agent1, message),
    Runner.run(sales_agent2, message),
    Runner.run(sales_agent3, message)
)

outputs = [result.final_output for result in results]
emails = "Sales Emails: \n\n" + "\n\n Email: \n\n".join(outputs)
response = await Runner.run(sales_sender, emails)
print(f"Final response:\n{response.final_output}")

### Did that work?!

See the traces for more! This is a great way to debug. Smaller models might require more time and experimentation.

https://platform.openai.com/traces

## Part 3: Orchestrating by LLMs

### 3a: via Tools

The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.

The OpenAI Agents SDK gives a very simple way to do this.

This works best when the flow is:

Agent A -> Agent B -> Agent A

And for the classic "Planning Agent" situation.

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool1

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

tools

## And now it's time for our Sales Manager - our planning agent

In [58]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=gemini_model)


In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, input=task)
print(result.final_output)

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!! Also look in your Junk / Spam folder - after all, this is basically a spam message..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Unreliable results?</h2>
            <span style="color:#ff7800;">This is par for the course with Agentic AI, and particularly with Orchestrating by LLM.
            To resolve it, you'll need to experiment and iterate with the prompts. Particularly with smaller models, more experiments
            may be needed to get reliable outcomes.
            </span>
        </td>
    </tr>
</table>

## Part 3: Orchestrating by LLMs

### 3a: via Handoffs

I am not a fan of handoffs. They seem very unreliable. They're not used consistently by other frameworks.

Behind the scenes, OpenAI Agents SDK has implemented these with Tools anyway.

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

A -> B -> A

With handoffs, control passes across

A -> B

In [ ]:

instructions = """
You are a Sales Manager at ComplAI. You get your sales team to draft emails, then send them all to a sales picker.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Handoff to the sales sender to choose and send the best email.
"""

tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, handoffs=handoffs, model=MODEL_NAME)


In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

Note that handoffs can be unrealiable and a little bit frustrating. I needed to force the tool use otherwise this didn't work. If you don't get reliable behavior, try iterating on the prompts - or use a larger model. And enjoy the process; this is what Agentic AI is all about!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Try extending with some more orchestration - perhaps another iteration of the email to refine it. 
            Use Orchestrating via Code and Orchestrating via LLMs.  
            HARD CHALLENGE: switch to using a professional email provider like SendGrid or Resend. Then handle the user replying. 
            Then have the SDR respond to keep the conversation going!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

# Customer Support Agents — Three Orchestration Patterns

A minimal, educational project demonstrating **three different ways** to orchestrate multiple AI agents using the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python).

The same customer-support scenario is implemented three times — once per orchestration pattern — so you can compare them side by side.

---

## The Scenario

A simple **customer support** system with three agents:

- **Triage Agent** — receives the user message and decides where it should go
- **Billing Agent** — handles invoices, payments, and refunds
- **FAQ Agent** — handles general questions and product information

The user asks things like:

- *"Where is my last invoice?"* → should reach **Billing Agent**
- *"What are your support hours?"* → should reach **FAQ Agent**
- *"I want to change my password"* → should reach **FAQ Agent**

The **same three agents** are reused across all three implementations. Only the orchestration layer changes.

---

## Three Orchestration Patterns

### 1. Code-based Orchestration

In this pattern, **your Python code owns the flow**. You run a classifier agent first, read its structured output, then decide in code which specialist to invoke.

**Characteristics**

- ✅ Fully **deterministic** — you can unit-test the routing logic
- ✅ Easy to add **guardrails**, retries, logging, and metrics
- ✅ Works well for **fixed pipelines** (classification → enrichment → response)
- ❌ Less flexible: the LLM cannot improvise a new route
- ❌ You must anticipate every branch in code

**When to use it**

- The flow is stable and predictable
- You need auditability (e.g. regulated domains)
- You want to compose agents with plain Python logic (`asyncio.gather`, loops, retries)

---

### 2. Agents as Tools (`agent.as_tool()`)

Here a **Manager Agent** keeps control of the conversation and calls specialists as if they were ordinary tools. The specialist answers the Manager — **not the user** — and the Manager writes the final response.

**Characteristics**

- ✅ The Manager stays in charge — one consistent **tone of voice**
- ✅ Easy to enforce **global guardrails** (e.g. always add a disclaimer)
- ✅ The Manager can **combine** answers from multiple specialists
- ✅ Each specialist call appears as a nested span in the trace
- ❌ The user never talks to the specialist directly
- ❌ Slightly more tokens: the Manager re-summarises everything

**When to use it**

- You want a **single point of contact** for the user
- The final answer must merge information from several sources
- You need consistent formatting / branding across all replies

---

### 3. Handoffs


The Triage Agent **transfers ownership** of the conversation to a specialist. From that point on, the specialist talks to the user directly, with its own instructions and personality.

**Characteristics**

- ✅ **True specialisation** — each agent has full control of its turn
- ✅ The specialist's own instructions, tools, and style apply
- ✅ Cleaner for long conversations that stay in one domain
- ✅ The SDK exposes handoffs to the LLM as `transfer_to_<agent_name>` tools
- ❌ Less control from the developer's side (the LLM decides)
- ❌ Harder to enforce a single global tone

**When to use it**

- The user's intent clearly belongs to one domain
- You want the specialist to own the rest of the conversation
- You are building an **expert routing** system (like a phone menu)

---

## Comparison

| Aspect | Code-based | Agents as Tools | Handoffs |
|---|---|---|---|
| **Flow controller** | Python code | Manager Agent | LLM (Triage) |
| **Final answer owner** | Python code | Manager | Specialist |
| **Determinism** | High | Medium | Low |
| **Flexibility** | Low | Medium | High |
| **Token cost** | Low | Medium–High | Medium |
| **Observability** | Easy (print/log in code) | Nested tool spans | Handoff events + `on_handoff` |
| **Best for** | Fixed pipelines | Answer synthesis | Expert routing |



### Customer Support Agent

In [1]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, Agent, Runner, function_tool, ModelSettings

load_dotenv(override=True)

gemini_api_key = os.getenv("GOOGLE_API_KEY")
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(api_key=gemini_api_key, base_url=gemini_base_url)
gemini_model = OpenAIChatCompletionsModel(openai_client=gemini_client, model="gemini-2.5-flash-lite")

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_base_url = "https://openrouter.ai/api/v1"
openrouter_client = AsyncOpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)
openrouter_model = OpenAIChatCompletionsModel(openai_client=openrouter_client, model="cohere/north-mini-code:free")

#### Orchestrate with code

In [2]:
triage_instruction = """
You are at the reception of the Lamaina store, a shop that sells stationery to customers.
You receive a customer's inquiry and determine whether it falls under the category of Billing requests or frequently asked questions (FAQs).
Respond using only one word: "Billing" or "FAQ".
"""

Billing_instruction = """
You are a financial specialist; answer only questions related to finance, customer payments, or refunds.
"""

FAQ_instruction = """
You are an expert on frequently asked questions and answer customers' common inquiries.
"""

triage_agent = Agent("Triage Agent", instructions=triage_instruction, model=openrouter_model)
billing_agent = Agent("Billing Agent", instructions=Billing_instruction, model=openrouter_model)
FAQ_agent = Agent("FAQ Agent", instructions=FAQ_instruction, model=openrouter_model)

In [3]:
async def chat (user_message: str):
    result  = await Runner.run(triage_agent, user_message)
    descision = result.final_output.strip().lower()

    if "billing" in descision:
        specialist = billing_agent
    elif "FAQ" in descision:
        specialist = FAQ_agent
    
    response = await Runner.run(specialist, user_message)
    return response.final_output

In [ ]:
# response = await chat("Hi, Where is my last month's invoice?")
response = await chat("Hi, What are the support hours?")
response

#### Orchestrate with LLM tools

In [10]:
Billing_instruction = """
You are a financial specialist; answer only questions related to finance, customer payments, or refunds.
"""

FAQ_instruction = """
You are an expert on frequently asked questions and answer customers' common inquiries.
"""

manager_instruction = """
You are a store support manager who receives requests and inquiries from customers; based on these, you must use the appropriate tools to provide answers.
Use "Billing tool" for financial inquiries.
Use "FAQ tool" for frequently asked questions.
Finally, generate the final response based on the tools and present it to the user.
"""
billing_agent = Agent("Billing Agent", instructions=Billing_instruction, model=openrouter_model)
FAQ_agent = Agent("FAQ Agent", instructions=FAQ_instruction, model=openrouter_model)

In [11]:
billing_tool = billing_agent.as_tool(tool_name="Billing_Tool", tool_description="For questions regarding finances, invoices, and payments")
FAQ_tool = FAQ_agent.as_tool(tool_name="FAQ_Tool", tool_description="For frequently asked and general questions")

tools = [billing_tool, FAQ_tool]
require_tool = ModelSettings(tool_choice="required")

manager_agent = Agent("Manager Agent", instructions=manager_instruction, model=openrouter_model, tools=tools, model_settings=require_tool)

In [12]:
async def chat(user_message: str):
    result = await Runner.run(manager_agent, user_message)
    return result.final_output

In [ ]:
# response = await chat("Hi, Where is my last month's invoice?")
response = await chat("Hi, What are the support hours?")
response

#### Orchestrate with LLM Handoffs

In [19]:
triage_instruction = """
You are at the reception of the Lamaina store, a shop that sells stationery to customers.
You receive a customer's inquiry. Handoff to the Billing Agent if the request or inquiry relates to financial matters and Handoff to the FAQ Agent for frequently asked questions.
"""

Billing_instruction = """
You are a financial specialist; answer only questions related to finance, customer payments, or refunds.
"""

FAQ_instruction = """
You are an expert on frequently asked questions and answer customers' common inquiries.
"""

billing_agent = Agent("Billing_Agent", instructions=Billing_instruction, model=openrouter_model)
FAQ_agent = Agent("FAQ_Agent", instructions=FAQ_instruction, model=openrouter_model)
triage_agent = Agent("Triage_Agent", instructions=triage_instruction, model=openrouter_model, handoffs=[billing_agent, FAQ_agent])

In [16]:
async def chat(user_message: str):
    result = await Runner.run(triage_agent, user_message)
    return result.final_output

In [ ]:
# response = await chat("Hi, Where is my last month's invoice?")
response = await chat("Hi, What are the support hours?")
response